In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
np.random.seed(42)


In [7]:
FORECAST_PATH = Path("../artifacts/weekly_forecast_future.csv")
assert FORECAST_PATH.exists(), f"Forecast file not found at {FORECAST_PATH}"

forecast_df = pd.read_csv(FORECAST_PATH)

required_cols = {"sku_id", "week", "p10", "p50", "p90"}
missing = required_cols - set(forecast_df.columns)
assert not missing, f"Forecast CSV missing columns: {missing}"

forecast_df.sort_values(["sku_id", "week"], inplace=True)
forecast_df.head()


,p50,p10,p90,week,sku_id,model_type,model_version
0,448.192962,421.833393,476.195795,1972-01-09,SKU0001,SARIMAX,v1.0.0
1,447.713938,418.105698,479.413889,1972-01-16,SKU0001,SARIMAX,v1.0.0
2,433.986245,405.276478,464.724805,1972-01-23,SKU0001,SARIMAX,v1.0.0
3,466.999266,436.110569,500.070732,1972-01-30,SKU0001,SARIMAX,v1.0.0
4,458.999964,428.639234,491.506155,1972-02-06,SKU0001,SARIMAX,v1.0.0


In [8]:
inventory = pd.DataFrame({
    "sku_id": ["SKU0001", "SKU0002", "SKU0003"],
    "warehouse_id": ["WH1", "WH1", "WH2"],
    "on_hand": [420, 120, 60],
    "on_order": [200, 0, 0],
    "last_updated_week": [103, 103, 103]
})

inventory


,sku_id,warehouse_id,on_hand,on_order,last_updated_week
0,SKU0001,WH1,420,200,103
1,SKU0002,WH1,120,0,103
2,SKU0003,WH2,60,0,103


In [9]:
vendors = pd.DataFrame({
    "sku_id": ["SKU0001", "SKU0002", "SKU0003"],
    "lead_time_weeks": [2, 3, 1],
    "MOQ": [500, 300, 200],
    "service_level": [0.9, 0.9, 0.9]
})

vendors


,sku_id,lead_time_weeks,MOQ,service_level
0,SKU0001,2,500,0.9
1,SKU0002,3,300,0.9
2,SKU0003,1,200,0.9


In [10]:
def route_sku(history_weeks: int) -> str:
    if history_weeks >= 60:
        return "SARIMAX"
    elif history_weeks >= 12:
        return "ML_BASELINE"
    else:
        return "NAIVE"


In [11]:
sku_history = (
    forecast_df
    .groupby("sku_id")["week"]
    .count()
    .rename("history_weeks")
    .reset_index()
)

sku_history["forecast_strategy"] = sku_history["history_weeks"].apply(route_sku)
sku_history


,sku_id,history_weeks,forecast_strategy
0,SKU0001,12,ML_BASELINE


In [12]:
context = (
    inventory
    .merge(vendors, on="sku_id", how="left")
    .merge(sku_history, on="sku_id", how="left")
)

context


,sku_id,warehouse_id,on_hand,on_order,last_updated_week,lead_time_weeks,MOQ,service_level,history_weeks,forecast_strategy
0,SKU0001,WH1,420,200,103,2,500,0.9,12.0,ML_BASELINE
1,SKU0002,WH1,120,0,103,3,300,0.9,NaN,NaN
2,SKU0003,WH2,60,0,103,1,200,0.9,NaN,NaN


In [13]:
context["inventory_position"] = context["on_hand"] + context["on_order"]
context


,sku_id,warehouse_id,on_hand,on_order,last_updated_week,lead_time_weeks,MOQ,service_level,history_weeks,forecast_strategy,inventory_position
0,SKU0001,WH1,420,200,103,2,500,0.9,12.0,ML_BASELINE,620
1,SKU0002,WH1,120,0,103,3,300,0.9,NaN,NaN,120
2,SKU0003,WH2,60,0,103,1,200,0.9,NaN,NaN,60


In [14]:
forecast_with_lt = forecast_df.merge(
    context[["sku_id", "lead_time_weeks"]],
    on="sku_id",
    how="left"
)

forecast_with_lt.head()


,p50,p10,p90,week,sku_id,model_type,model_version,lead_time_weeks
0,448.192962,421.833393,476.195795,1972-01-09,SKU0001,SARIMAX,v1.0.0,2
1,447.713938,418.105698,479.413889,1972-01-16,SKU0001,SARIMAX,v1.0.0,2
2,433.986245,405.276478,464.724805,1972-01-23,SKU0001,SARIMAX,v1.0.0,2
3,466.999266,436.110569,500.070732,1972-01-30,SKU0001,SARIMAX,v1.0.0,2
4,458.999964,428.639234,491.506155,1972-02-06,SKU0001,SARIMAX,v1.0.0,2


In [15]:
expected_lt_demand = (
    forecast_with_lt
    .groupby("sku_id")
    .apply(
        lambda df: df.head(int(df["lead_time_weeks"].iloc[0]))["p50"].sum()
    )
    .rename("expected_demand_LT")
    .reset_index()
)

expected_lt_demand


C:\Users\ACER\AppData\Local\Temp\ipykernel_19228\643507821.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


,sku_id,expected_demand_LT
0,SKU0001,895.9069


In [16]:
safety_stock = (
    forecast_with_lt
    .groupby("sku_id")
    .apply(
        lambda df: (
            df.head(int(df["lead_time_weeks"].iloc[0]))["p90"].sum()
            - df.head(int(df["lead_time_weeks"].iloc[0]))["p50"].sum()
        )
    )
    .rename("safety_stock")
    .reset_index()
)

safety_stock


C:\Users\ACER\AppData\Local\Temp\ipykernel_19228\2005423058.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


,sku_id,safety_stock
0,SKU0001,59.702784


In [17]:
reorder_points = (
    expected_lt_demand
    .merge(safety_stock, on="sku_id")
)

reorder_points["reorder_point"] = (
    reorder_points["expected_demand_LT"] +
    reorder_points["safety_stock"]
)

reorder_points


,sku_id,expected_demand_LT,safety_stock,reorder_point
0,SKU0001,895.9069,59.702784,955.609684


In [18]:
decision_df = (
    context
    .merge(reorder_points, on="sku_id", how="left")
)

decision_df


,sku_id,warehouse_id,on_hand,on_order,last_updated_week,lead_time_weeks,MOQ,service_level,history_weeks,forecast_strategy,inventory_position,expected_demand_LT,safety_stock,reorder_point
0,SKU0001,WH1,420,200,103,2,500,0.9,12.0,ML_BASELINE,620,895.9069,59.702784,955.609684
1,SKU0002,WH1,120,0,103,3,300,0.9,NaN,NaN,120,NaN,NaN,NaN
2,SKU0003,WH2,60,0,103,1,200,0.9,NaN,NaN,60,NaN,NaN,NaN


In [19]:
decision_df["reorder_required"] = (
    decision_df["inventory_position"] <= decision_df["reorder_point"]
)

decision_df


,sku_id,warehouse_id,on_hand,on_order,last_updated_week,lead_time_weeks,MOQ,service_level,history_weeks,forecast_strategy,inventory_position,expected_demand_LT,safety_stock,reorder_point,reorder_required
0,SKU0001,WH1,420,200,103,2,500,0.9,12.0,ML_BASELINE,620,895.9069,59.702784,955.609684,True
1,SKU0002,WH1,120,0,103,3,300,0.9,NaN,NaN,120,NaN,NaN,NaN,False
2,SKU0003,WH2,60,0,103,1,200,0.9,NaN,NaN,60,NaN,NaN,NaN,False


In [20]:
def recommend_qty(row):
    if not row["reorder_required"]:
        return 0
    gap = row["reorder_point"] - row["inventory_position"]
    return max(
        row["MOQ"],
        int(np.ceil(gap / row["MOQ"]) * row["MOQ"])
    )

decision_df["recommended_order_qty"] = decision_df.apply(recommend_qty, axis=1)
decision_df


,sku_id,warehouse_id,on_hand,on_order,last_updated_week,lead_time_weeks,MOQ,service_level,history_weeks,forecast_strategy,inventory_position,expected_demand_LT,safety_stock,reorder_point,reorder_required,recommended_order_qty
0,SKU0001,WH1,420,200,103,2,500,0.9,12.0,ML_BASELINE,620,895.9069,59.702784,955.609684,True,500
1,SKU0002,WH1,120,0,103,3,300,0.9,NaN,NaN,120,NaN,NaN,NaN,False,0
2,SKU0003,WH2,60,0,103,1,200,0.9,NaN,NaN,60,NaN,NaN,NaN,False,0


In [21]:
def risk_label(row):
    if row["inventory_position"] < row["expected_demand_LT"]:
        return "HIGH"
    elif row["inventory_position"] < row["reorder_point"]:
        return "MEDIUM"
    else:
        return "LOW"

decision_df["risk_level"] = decision_df.apply(risk_label, axis=1)
decision_df


,sku_id,warehouse_id,on_hand,on_order,last_updated_week,lead_time_weeks,MOQ,service_level,history_weeks,forecast_strategy,inventory_position,expected_demand_LT,safety_stock,reorder_point,reorder_required,recommended_order_qty,risk_level
0,SKU0001,WH1,420,200,103,2,500,0.9,12.0,ML_BASELINE,620,895.9069,59.702784,955.609684,True,500,HIGH
1,SKU0002,WH1,120,0,103,3,300,0.9,NaN,NaN,120,NaN,NaN,NaN,False,0,LOW
2,SKU0003,WH2,60,0,103,1,200,0.9,NaN,NaN,60,NaN,NaN,NaN,False,0,LOW


In [22]:
final_output = decision_df[[
    "sku_id",
    "forecast_strategy",
    "inventory_position",
    "expected_demand_LT",
    "safety_stock",
    "reorder_point",
    "recommended_order_qty",
    "risk_level"
]]

final_output


,sku_id,forecast_strategy,inventory_position,expected_demand_LT,safety_stock,reorder_point,recommended_order_qty,risk_level
0,SKU0001,ML_BASELINE,620,895.9069,59.702784,955.609684,500,HIGH
1,SKU0002,NaN,120,NaN,NaN,NaN,0,LOW
2,SKU0003,NaN,60,NaN,NaN,NaN,0,LOW
